# Bird false positives

Run the full-frame drone detector on still images containing birds. An
**image alert** is any drone detection at or above the selected confidence.
When bird boxes are available, a **bird match** requires a drone box to overlap
an annotated bird by at least the selected intersection over union (IoU).
Bird matches are measured on the subset with annotations.

This test treats the photos as drone-free. The default sample excludes positive
drone labels and includes general bird photos such as perched birds. The image
alert rate counts alerts anywhere in a photo, while the bird match rate uses
the bird annotations. Video tracking, trajectory crops, and motion cues require
a sequence.

From the project root, open this notebook with
`uv run jupyter lab notebooks/bird_false_positives.ipynb` and select
**Python (Drone Tracking v2)**. The [dataset notes](../bird_data/README.md) describe
the local Open Images sample and its source.

In [ ]:
import json
import random
import sys
import tempfile
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from time import perf_counter

import cv2
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display
from ultralytics import YOLO, __version__ as ultralytics_version

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "run_video.py").is_file():
    raise RuntimeError("Open the notebook from the project root or its notebooks folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import mlayer  # Some local checkpoints reference this module.

LEGACY_ROOT = ROOT.parent / "CF Drone trecking "
SKYGUARD_WEIGHTS = ROOT.parent / "pretrained" / "drone-detection-yolov11" / "best.pt"
if not SKYGUARD_WEIGHTS.is_file():
    SKYGUARD_WEIGHTS = LEGACY_ROOT / "models" / "skyguard-best.pt"


@dataclass(frozen=True)
class Detector:
    weights: Path | str
    imgsz: int = 1280
    drone_labels: tuple[str, ...] = ("drone",)

## Configure the test

`IMAGE_DIR` can be any local folder of drone-free images. The optional bird-box
CSV has columns `image,xmin,ymin,xmax,ymax`, with paths relative to the image
folder and coordinates normalized to the image dimensions. Set
`BIRD_BOXES = None` to measure image alerts without labels.

Adjust the models, confidence thresholds and IoU below. The detector runs
once per image and model, at a confidence below the smallest threshold.
A batch size of one matches the per-frame inference in the video pipeline
and supports fixed-batch ONNX exports.

In [ ]:
IMAGE_DIR = ROOT / "bird_data" / "images"
BIRD_BOXES = IMAGE_DIR.parent / "bird_boxes.csv"
MODELS = {"skyguard": Detector(SKYGUARD_WEIGHTS)}
# Compare a local 640-pixel ONNX model by adding:
# MODELS["skyguard-onnx-640"] = Detector(
#     LEGACY_ROOT / "models" / "skyguard-yolo11n.onnx", imgsz=640
# )

DEVICE = "cpu"  # Use "" for auto selection, "mps" for Metal or a CUDA device.
BATCH_SIZE = 1
THRESHOLDS = (0.01, 0.05, 0.10, 0.30, 0.60)
MATCH_IOU = 0.50
MAX_IMAGES = None  # Set a positive number to take a seeded random subset.
RANDOM_SEED = 42
PREVIEW_THRESHOLD = 0.10
PREVIEWS_PER_MODEL = 6
OUTPUT_ROOT = ROOT / "preds" / "bird_false_positives"

In [3]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
BOX_COLUMNS = ["xmin", "ymin", "xmax", "ymax"]
PREDICTION_COLUMNS = ["model", "image", "confidence", *BOX_COLUMNS, "max_bird_iou"]
IMAGE_COLUMNS = ["model", "image", "bird_boxes", "max_drone_confidence", "max_bird_confidence"]


def local_path(value):
    return (ROOT / Path(value).expanduser()).resolve()


def read_annotations(path):
    if path is None:
        return {}
    if not path.is_file():
        print(f"Bird-box CSV absent at {path}. Reporting image alerts only.")
        return {}

    records = pd.read_csv(path, dtype={"image": str})
    required = {"image", *BOX_COLUMNS}
    if not required.issubset(records.columns):
        raise ValueError(f"{path} must contain {sorted(required)}.")
    coordinates = records[BOX_COLUMNS].apply(pd.to_numeric).to_numpy(dtype=float)
    if (
        records["image"].isna().any()
        or not np.isfinite(coordinates).all()
        or ((coordinates < 0) | (coordinates > 1)).any()
        or (coordinates[:, :2] >= coordinates[:, 2:]).any()
    ):
        raise ValueError("Bird boxes need image paths and valid normalized XYXY coordinates.")
    records[BOX_COLUMNS] = coordinates
    return {
        image: group[BOX_COLUMNS].to_numpy(dtype=float)
        for image, group in records.groupby("image", sort=False)
    }


def prepare():
    image_root = local_path(IMAGE_DIR)
    if not image_root.is_dir():
        raise ValueError(f"Set IMAGE_DIR to a folder of local bird images. Tried {image_root}.")
    paths = sorted(
        path.relative_to(image_root).as_posix()
        for path in image_root.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
        and not any(part.startswith(".") for part in path.relative_to(image_root).parts)
    )
    if not paths:
        raise ValueError(f"No supported images found in {image_root}.")
    if MAX_IMAGES is not None:
        if type(MAX_IMAGES) is not int or MAX_IMAGES < 1:
            raise ValueError("MAX_IMAGES must be a positive integer or None.")
        if MAX_IMAGES < len(paths):
            paths = sorted(random.Random(RANDOM_SEED).sample(paths, MAX_IMAGES))

    thresholds = sorted(set(THRESHOLDS))
    if not thresholds or not all(0 < threshold <= 1 for threshold in [*thresholds, PREVIEW_THRESHOLD]):
        raise ValueError("Set confidence thresholds in (0, 1].")
    if not 0 < MATCH_IOU <= 1:
        raise ValueError("MATCH_IOU must be in (0, 1].")
    if type(BATCH_SIZE) is not int or BATCH_SIZE < 1:
        raise ValueError("BATCH_SIZE must be a positive integer.")
    if not MODELS:
        raise ValueError("Configure at least one detector in MODELS.")
    for name, spec in MODELS.items():
        if not local_path(spec.weights).is_file():
            raise ValueError(f"{name}: weights not found at {local_path(spec.weights)}.")
        if spec.imgsz < 1 or not spec.drone_labels:
            raise ValueError(f"{name}: configure a positive image size and at least one drone label.")

    csv_path = local_path(BIRD_BOXES) if BIRD_BOXES is not None else None
    annotations = read_annotations(csv_path)
    print(f"{len(paths)} selected images · {sum(path in annotations for path in paths)} with bird boxes")
    print(f"Images: {image_root}")
    for name, spec in MODELS.items():
        print(f"{name}: {local_path(spec.weights)} ({spec.imgsz} px)")
    return image_root, paths, csv_path, annotations, thresholds


INPUTS = None
try:
    INPUTS = prepare()
except (OSError, ValueError) as error:
    print(f"Update the configuration to continue. {error}")

200 selected images · 200 with bird boxes
Images: /Users/romaniliuchenko/Documents/Code/Drone Tracking v2/bird_data/images
skyguard: /Users/romaniliuchenko/Documents/Code/CF Drone trecking /models/skyguard-best.pt (1280 px)


## Run the detector

The notebook saves the image list, model settings, per-image outcomes, drone
boxes and threshold summary in a new folder under `preds/bird_false_positives/`.
Unreadable files are listed and excluded from the rates.

In [4]:
def best_ious(detections, bird_boxes):
    if len(bird_boxes) == 0:
        return np.full(len(detections), np.nan)
    left_top = np.maximum(detections[:, None, :2], bird_boxes[None, :, :2])
    right_bottom = np.minimum(detections[:, None, 2:], bird_boxes[None, :, 2:])
    intersection = np.clip(right_bottom - left_top, 0, None).prod(axis=2)
    detection_area = np.clip(detections[:, 2:] - detections[:, :2], 0, None).prod(axis=1)
    bird_area = (bird_boxes[:, 2:] - bird_boxes[:, :2]).prod(axis=1)
    union = detection_area[:, None] + bird_area[None, :] - intersection
    return (intersection / union).max(axis=1)


def run_detector(name, spec, image_root, paths, annotations, cutoff, device):
    model = YOLO(str(local_path(spec.weights)))
    wanted = {label.strip().casefold() for label in spec.drone_labels}
    classes = {index: label for index, label in model.names.items() if label.casefold() in wanted}
    missing = wanted - {label.casefold() for label in classes.values()}
    if missing:
        raise ValueError(f"{name}: drone labels {sorted(missing)} not in {model.names}.")

    observations, predictions, unreadable = [], [], []
    start = perf_counter()
    print(f"Running {name} on {device} …", flush=True)
    for offset in range(0, len(paths), BATCH_SIZE):
        batch_paths = paths[offset:offset + BATCH_SIZE]
        readable, images = [], []
        for path in batch_paths:
            try:
                image = cv2.imread(str(image_root / path), cv2.IMREAD_COLOR)
            except cv2.error:
                image = None
            if image is None:
                unreadable.append(path)
            else:
                readable.append(path)
                images.append(image)
        if images:
            results = model.predict(
                images, imgsz=spec.imgsz, conf=cutoff, classes=list(classes),
                device=device, verbose=False,
            )
            for path, result in zip(readable, results, strict=True):
                boxes = result.boxes.xyxyn.cpu().numpy()
                confidence = result.boxes.conf.cpu().numpy()
                birds = annotations.get(path, ())
                overlaps = best_ious(boxes, birds)
                predictions.extend(
                    (name, path, float(score), *map(float, box), float(overlap))
                    for box, score, overlap in zip(boxes, confidence, overlaps, strict=True)
                )
                matched = confidence[overlaps >= MATCH_IOU]
                observations.append((
                    name, path, len(birds),
                    float(confidence.max()) if confidence.size else np.nan,
                    float(matched.max()) if matched.size else np.nan,
                ))
        attempted = min(offset + BATCH_SIZE, len(paths))
        if attempted == len(paths) or attempted % 50 < BATCH_SIZE:
            print(f"  {attempted}/{len(paths)} files checked", flush=True)

    print(f"  {len(observations)} read, {len(unreadable)} unreadable, {perf_counter() - start:.1f} s")
    return observations, predictions, unreadable


def summarize(images, thresholds):
    rows = []
    for name, evaluated in images.groupby("model", sort=False):
        labeled = evaluated["bird_boxes"] > 0
        for threshold in thresholds:
            alerts = int(evaluated["max_drone_confidence"].ge(threshold).sum())
            bird_hits = int(evaluated.loc[labeled, "max_bird_confidence"].ge(threshold).sum())
            rows.append({
                "model": name, "confidence": threshold, "images": len(evaluated),
                "image_alerts": alerts, "image_alert_rate_pct": 100 * alerts / len(evaluated),
                "annotated_images": int(labeled.sum()), "bird_matches": bird_hits,
                "bird_match_rate_pct": 100 * bird_hits / labeled.sum() if labeled.any() else np.nan,
            })
    return pd.DataFrame(rows)


def run_test(inputs):
    image_root, paths, csv_path, annotations, thresholds = inputs
    device = DEVICE or ("mps" if torch.backends.mps.is_available() else
                        "cuda" if torch.cuda.is_available() else "cpu")
    cutoff = min(*thresholds, PREVIEW_THRESHOLD) / 2
    observations, predictions, unreadable_rows = [], [], []
    for name, spec in MODELS.items():
        evaluated, detections, unreadable = run_detector(
            name, spec, image_root, paths, annotations, cutoff, device,
        )
        if not evaluated:
            raise RuntimeError(f"{name}: none of the selected images could be opened.")
        observations.extend(evaluated)
        predictions.extend(detections)
        unreadable_rows.extend((name, path) for path in unreadable)

    images = pd.DataFrame(observations, columns=IMAGE_COLUMNS)
    boxes = pd.DataFrame(predictions, columns=PREDICTION_COLUMNS)
    summary = summarize(images, thresholds)
    output_root = local_path(OUTPUT_ROOT)
    output_root.mkdir(parents=True, exist_ok=True)
    prefix = f"{datetime.now():%Y%m%d-%H%M%S}-"
    run_dir = Path(tempfile.mkdtemp(prefix=prefix, dir=output_root))
    settings = {
        "image_dir": str(image_root), "selected_images": paths,
        "bird_boxes_csv": str(csv_path) if csv_path and csv_path.is_file() else None,
        "models": {
            name: {**asdict(spec), "weights": str(local_path(spec.weights))}
            for name, spec in MODELS.items()
        },
        "device": device, "batch_size": BATCH_SIZE, "inference_confidence": cutoff,
        "thresholds": thresholds, "bird_iou": MATCH_IOU, "seed": RANDOM_SEED,
        "ultralytics": ultralytics_version,
    }
    (run_dir / "settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
    images.to_csv(run_dir / "images.csv", index=False)
    boxes.to_csv(run_dir / "predictions.csv", index=False)
    pd.DataFrame(unreadable_rows, columns=["model", "image"]).to_csv(run_dir / "unreadable.csv", index=False)
    summary.to_csv(run_dir / "summary.csv", index=False)
    print(f"Results: {run_dir}")
    return image_root, run_dir, annotations, boxes, summary


RESULTS = run_test(INPUTS) if INPUTS is not None else None

Running skyguard on cpu …


  50/200 files checked


  100/200 files checked


  150/200 files checked


  200/200 files checked


  200 read, 0 unreadable, 14.2 s
Results: /Users/romaniliuchenko/Documents/Code/Drone Tracking v2/preds/bird_false_positives/20261121-102302-2lanm776


## Read the results

The image alert rate counts each readable image once, even when the detector
draws several boxes. The bird match rate counts each annotated image once
when any drone box meets the IoU threshold for any bird. Review the examples
to see whether alerts correspond to birds or other objects in the scene.
Rates depend on the dataset's bird sizes and scene types.

In [5]:
if RESULTS is not None:
    display(RESULTS[-1].round(3))

,model,confidence,images,image_alerts,image_alert_rate_pct,annotated_images,bird_matches,bird_match_rate_pct
0,skyguard,0.01,200,197,98.5,200,61,30.5
1,skyguard,0.05,200,177,88.5,200,41,20.5
2,skyguard,0.10,200,159,79.5,200,35,17.5
3,skyguard,0.30,200,86,43.0,200,25,12.5
4,skyguard,0.60,200,21,10.5,200,13,6.5


## Inspect the highest-scoring alerts

Orange boxes are drone predictions and green boxes are annotated birds.
Examples with a matching bird appear first, followed by other alerts ordered
by confidence. The same annotated previews are saved alongside the CSV files.

In [ ]:
def annotate(frame, boxes, labels, color, *, below=False):
    height, width = frame.shape[:2]
    for box, label in zip(boxes, labels, strict=True):
        x1, y1, x2, y2 = np.rint(box * [width, height, width, height]).astype(int)
        x1, x2 = np.clip([x1, x2], 0, width - 1)
        y1, y2 = np.clip([y1, y2], 0, height - 1)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        (text_width, text_height), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        text_x = min(x1, max(0, width - text_width - 3))
        text_y = y2 + text_height + 5 if below else y1 - 5
        text_y = min(max(text_y, text_height + 2), height - 3)
        point = (text_x, text_y)
        cv2.putText(frame, label, point, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 3)
        cv2.putText(frame, label, point, cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)


def show_previews(results):
    image_root, run_dir, annotations, predictions, _ = results
    preview_dir = run_dir / "previews"
    for model_index, name in enumerate(MODELS, start=1):
        alerts = predictions.loc[
            (predictions["model"] == name) & predictions["confidence"].ge(PREVIEW_THRESHOLD)
        ].copy()
        if alerts.empty:
            print(f"{name}: no alerts at confidence {PREVIEW_THRESHOLD:g}.")
            continue
        alerts["bird_match"] = alerts["max_bird_iou"].ge(MATCH_IOU)
        ranked = alerts.sort_values(["bird_match", "confidence"], ascending=False)
        examples = ranked.drop_duplicates("image").head(PREVIEWS_PER_MODEL)
        preview_dir.mkdir(exist_ok=True)
        for sample_index, example in enumerate(examples.itertuples(index=False), start=1):
            frame = cv2.imread(str(image_root / example.image), cv2.IMREAD_COLOR)
            if frame is None:
                print(f"Preview image unavailable: {example.image}")
                continue
            if frame.shape[1] > 1000:
                scale = 1000 / frame.shape[1]
                frame = cv2.resize(frame, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
            birds = annotations.get(example.image, ())
            annotate(frame, birds, ["Bird"] * len(birds), (75, 230, 95), below=True)
            detected = alerts.loc[alerts["image"] == example.image]
            labels = [f"Drone {score:.2f}" for score in detected["confidence"]]
            annotate(frame, detected[BOX_COLUMNS].to_numpy(dtype=float), labels, (0, 160, 255))
            ok, jpeg = cv2.imencode(".jpg", frame)
            if not ok:
                raise RuntimeError(f"Could not render preview for {example.image}.")
            data = jpeg.tobytes()
            (preview_dir / f"{model_index:02d}-{sample_index:02d}.jpg").write_bytes(data)
            print(f"{name} · {example.image} · confidence {example.confidence:.3f}")
            display(Image(data=data, format="jpeg"))


if RESULTS is not None:
    show_previews(RESULTS)